# An AgentCore Gateway over the UnderCover app

The UnderCover app is on Lambda, with an API key on its claims routes. This notebook puts an AgentCore Gateway in front of it. The Gateway reads the app's OpenAPI document and offers each operation as an MCP tool, so an agent can work with UnderCover's claims without knowing the app's address or holding its key.

What it creates, all named `undercover` and tagged `project=undercover`:

- a credential provider, `undercover-app-key`, which keeps the app's API key in AgentCore Identity
- an IAM role, `undercover-gateway-role`, that the gateway runs as
- a gateway, `undercover-gateway`, with one target, `undercover-app`
- a policy engine, `undercover_policy_engine`, with two rules, one of which stops any payout over $1,000

Callers sign their requests with their AWS credentials, so there is no user pool to set up. Nothing here bills by the hour. Gateway calls cost half a cent per thousand, a policy decision costs a fraction of that, and the secret behind the API key costs 40 cents a month, prorated, so a lab torn down the same day costs under a cent.

Run `lambda_deploy.ipynb` first. This notebook finds the app by its function name.

## 1. Setup

boto3 reads the same credentials as the AWS CLI. `bedrock-agentcore-control` is the client that creates and deletes AgentCore resources.

In [ ]:
import json
import os
import time

import boto3
import requests
from dotenv import load_dotenv

PREFIX = "undercover"
FUNCTION_NAME = "undercover-app"
KEY_PROVIDER_NAME = "undercover-app-key"
ROLE_NAME = "undercover-gateway-role"
GATEWAY_NAME = "undercover-gateway"
TARGET_NAME = "undercover-app"
PROJECT_TAG = {"project": "undercover"}

session = boto3.Session()
region = session.region_name
agentcore = session.client("bedrock-agentcore-control")
iam = session.client("iam")
lambda_client = session.client("lambda")
logs = session.client("logs")
secrets_manager = session.client("secretsmanager")
tagging = session.client("resourcegroupstaggingapi")

identity = session.client("sts").get_caller_identity()
account = identity["Account"]
print(f"Working as {identity['Arn']} in {region}")

A look at the budget before anything is created. The Budgets API is free to call.

In [ ]:
budgets = session.client("budgets").describe_budgets(AccountId=account).get("Budgets", [])

for budget in budgets:
    limit = float(budget["BudgetLimit"]["Amount"])
    actual = float(budget["CalculatedSpend"]["ActualSpend"]["Amount"])
    print(f"{budget['BudgetName']}: ${actual:,.2f} spent of ${limit:,.2f}")

if not budgets:
    print("No budget found. Create one in Billing and Cost Management, Budgets, before going further.")

The app is found by its function name, so no address is pasted between notebooks. The API key comes from `.env`, where the Lambda notebook put it.

In [ ]:
load_dotenv()
API_KEY = os.environ["UNDERCOVER_API_KEY"]

app_url = lambda_client.get_function_url_config(FunctionName=FUNCTION_NAME)["FunctionUrl"].rstrip("/")
response = requests.get(f"{app_url}/claims/CLM-1002", headers={"X-API-Key": API_KEY}, timeout=30)

print(app_url)
print(response.status_code, response.json())

## 2. The API key goes into AgentCore Identity

The Gateway will call the app on the agent's behalf, so it needs the app's API key. A credential provider holds the key. AgentCore Identity stores it as a secret in AWS Secrets Manager, and from here on the key is referred to by the provider's ARN. Running the cell again updates the stored key to match `.env`.

In [ ]:
def find_key_providers():
    pages = agentcore.get_paginator("list_api_key_credential_providers").paginate()
    return [p["name"] for page in pages for p in page["credentialProviders"] if p["name"].startswith(PREFIX)]


if KEY_PROVIDER_NAME in find_key_providers():
    agentcore.update_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY)
    print(f"Credential provider {KEY_PROVIDER_NAME} already exists, key updated")
else:
    agentcore.create_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY, tags=PROJECT_TAG)
    print(f"Created credential provider {KEY_PROVIDER_NAME}")

provider = agentcore.get_api_key_credential_provider(name=KEY_PROVIDER_NAME)
provider_arn = provider["credentialProviderArn"]
secret_arn = provider["apiKeySecretArn"]["secretArn"]

print(provider_arn)
print(secret_arn)

## 3. The role the gateway runs as

The trust policy lets the AgentCore service assume the role, and only for a gateway in this account whose name starts with `undercover-gateway`. The permission policy allows the three things the gateway does to fetch the API key: get a workload token for itself, ask AgentCore Identity for the key, and read the secret behind it. Each statement names the exact resources. AgentCore checks the request for the key against the provider, the vault that holds it and the gateway's own workload identity, so the second statement lists all of them.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account}:gateway/{GATEWAY_NAME}-*"},
            },
        }
    ],
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="The role the UnderCover gateway runs as",
        Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
    )["Role"]
    print(f"Created role {ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    role = iam.get_role(RoleName=ROLE_NAME)["Role"]
    print(f"Role {ROLE_NAME} already exists")

workload_directory = f"arn:aws:bedrock-agentcore:{region}:{account}:workload-identity-directory/default"
workload_identity = [workload_directory, f"{workload_directory}/workload-identity/{GATEWAY_NAME}-*"]
token_vault = provider_arn.split("/apikeycredentialprovider/")[0]

api_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "GetWorkloadAccessToken",
            "Effect": "Allow",
            "Action": "bedrock-agentcore:GetWorkloadAccessToken",
            "Resource": workload_identity,
        },
        {
            "Sid": "GetResourceApiKey",
            "Effect": "Allow",
            "Action": "bedrock-agentcore:GetResourceApiKey",
            "Resource": [provider_arn, token_vault, *workload_identity],
        },
        {
            "Sid": "GetSecretValue",
            "Effect": "Allow",
            "Action": "secretsmanager:GetSecretValue",
            "Resource": secret_arn,
        },
    ],
}

iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="undercover-gateway-api-key", PolicyDocument=json.dumps(api_key_policy))
print("The role can fetch the app's API key")

## 4. The gateway

`AWS_IAM` as the authorizer means a caller signs each request with AWS credentials, the way boto3 signs every call in this notebook, and needs the `bedrock-agentcore:InvokeGateway` permission. `exceptionLevel="DEBUG"` makes the gateway's error messages say what went wrong. A brand new role takes a few seconds to reach AgentCore, so the create call retries.

In [ ]:
def find_gateways(prefix=GATEWAY_NAME):
    pages = agentcore.get_paginator("list_gateways").paginate()
    return [g for page in pages for g in page["items"] if g["name"].startswith(prefix)]


def wait_for(get_resource, label):
    while True:
        resource = get_resource()
        if resource["status"] not in ("CREATING", "UPDATING", "SYNCHRONIZING"):
            break
        time.sleep(3)
    print(f"{label}: {resource['status']}")
    for reason in resource.get("statusReasons", []):
        print(f"  {reason}")
    return resource


def create_gateway():
    for attempt in range(10):
        try:
            return agentcore.create_gateway(
                name=GATEWAY_NAME,
                description="UnderCover's systems as tools for the agent",
                roleArn=role["Arn"],
                protocolType="MCP",
                authorizerType="AWS_IAM",
                exceptionLevel="DEBUG",
                tags=PROJECT_TAG,
            )
        except agentcore.exceptions.ValidationException as error:
            if "role" not in str(error).lower():
                raise
            print("Waiting for the new role to reach AgentCore")
            time.sleep(3)
    raise RuntimeError("AgentCore could not use the role")


existing = find_gateways()
gateway_id = existing[0]["gatewayId"] if existing else create_gateway()["gatewayId"]

gateway = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
gateway_arn = gateway["gatewayArn"]
gateway_url = gateway["gatewayUrl"]
print(gateway_url)

## 5. The app becomes a target

FastAPI writes the app's OpenAPI document, with an `operationId` and a description on every operation. The Gateway turns each operation into a tool and uses those two fields as the tool's name and description, which is why `server.py` sets them with care. FastAPI leaves out the `servers` entry, since an app does not know its own public address, so the cell adds the Function URL before handing the document over.

The credential configuration tells the Gateway to send the key from the provider in the `X-API-Key` header of every call to the app.

In [ ]:
openapi = requests.get(f"{app_url}/openapi.json", timeout=30).json()
openapi["servers"] = [{"url": app_url}]

for path, operations in openapi["paths"].items():
    for method, operation in operations.items():
        print(f"{method.upper():<5}{path:<28}{operation['operationId']:<16}{operation['description']}")

In [ ]:
def find_targets(gateway_id):
    pages = agentcore.get_paginator("list_gateway_targets").paginate(gatewayIdentifier=gateway_id)
    return [t for page in pages for t in page["items"]]


target_settings = {
    "gatewayIdentifier": gateway_id,
    "name": TARGET_NAME,
    "description": "The UnderCover app's API",
    "targetConfiguration": {"mcp": {"openApiSchema": {"inlinePayload": json.dumps(openapi)}}},
    "credentialProviderConfigurations": [
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": provider_arn,
                    "credentialParameterName": "X-API-Key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
}

existing = [t for t in find_targets(gateway_id) if t["name"] == TARGET_NAME]

if existing:
    target_id = existing[0]["targetId"]
    agentcore.update_gateway_target(targetId=target_id, **target_settings)
else:
    target_id = agentcore.create_gateway_target(**target_settings)["targetId"]

target = wait_for(
    lambda: agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id),
    f"Target {target_id}",
)

## 6. Call the tools

The gateway speaks MCP over HTTP, so any MCP client can use it as long as its requests are signed. `SigV4Auth` below signs each outgoing request with the same AWS credentials boto3 uses, for the service name `bedrock-agentcore`. The agent will use this same class later.

In [ ]:
from contextlib import asynccontextmanager

import httpx2
from botocore.auth import SigV4Auth as BotocoreSigV4Auth
from botocore.awsrequest import AWSRequest
from mcp import ClientSession, MCPError
from mcp.client.streamable_http import streamable_http_client


class SigV4Auth(httpx2.Auth):
    """Sign each request with AWS Signature Version 4, as boto3 does for its own calls."""

    requires_request_body = True

    def __init__(self, session, service="bedrock-agentcore"):
        self.credentials = session.get_credentials()
        self.region = session.region_name
        self.service = service

    def auth_flow(self, request):
        headers = {"Host": request.url.netloc.decode(), "Content-Type": request.headers.get("content-type", "")}
        aws_request = AWSRequest(method=request.method, url=str(request.url), data=request.content, headers=headers)
        BotocoreSigV4Auth(self.credentials.get_frozen_credentials(), self.service, self.region).add_auth(aws_request)
        for name in ("Authorization", "X-Amz-Date", "X-Amz-Security-Token"):
            if name in aws_request.headers:
                request.headers[name] = aws_request.headers[name]
        yield request

In [ ]:
@asynccontextmanager
async def gateway_session():
    """Open a signed MCP session with the gateway."""
    async with httpx2.AsyncClient(auth=SigV4Auth(session), timeout=60) as http_client:
        async with streamable_http_client(gateway_url, http_client=http_client) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as mcp_session:
                await mcp_session.initialize()
                yield mcp_session


async with gateway_session() as mcp_session:
    tools = (await mcp_session.list_tools()).tools

for tool in tools:
    print(f"{tool.name:<36}{tool.description}")

Each tool is named after the target and the operation. The input schema comes from the route's parameters and its JSON body, flattened into one object.

In [ ]:
for tool in tools:
    print(tool.name)
    print(json.dumps(tool.input_schema))

A brand new IAM role takes up to a minute to work everywhere in AWS. Until it does, the gateway cannot assume its role and a tool call comes back with an error about credentials. This cell calls the `health` tool until it answers.

In [ ]:
import asyncio

for attempt in range(20):
    async with gateway_session() as mcp_session:
        result = await mcp_session.call_tool(f"{TARGET_NAME}___health", {})
    if not result.is_error:
        break
    print(f"Waiting for the gateway's role: {result.content[0].text[:90]}")
    await asyncio.sleep(5)

print(result.content[0].text)

A tool call can end three ways: a result, an error from the app such as a 404, or a refusal by the gateway itself, which arrives as an `MCPError`.

In [ ]:
async def call_tool(operation, **arguments):
    """Call one of the app's operations through the gateway and print what comes back."""
    async with gateway_session() as mcp_session:
        try:
            result = await mcp_session.call_tool(f"{TARGET_NAME}___{operation}", arguments)
        except MCPError as error:
            print(f"refused   {error}")
            return
    for block in result.content:
        print(f"{'error' if result.is_error else 'ok':<10}{block.text}")


await call_tool("get_claim", claim_id="CLM-1002")
await call_tool("get_claim", claim_id="CLM-9999")

The first answer came from the Lambda function, which means the gateway sent the API key. The app's own log has a report line for each request.

In [ ]:
time.sleep(5)
pages = logs.get_paginator("filter_log_events").paginate(
    logGroupName=f"/aws/lambda/{FUNCTION_NAME}", filterPattern="REPORT", startTime=int((time.time() - 300) * 1000)
)
events = [event for page in pages for event in page["events"]][-4:]

for event in events:
    print(time.strftime("%H:%M:%S", time.localtime(event["timestamp"] / 1000)), event["message"].strip().replace("\t", "  ")[:110])

## 7. A rule on payouts

`approve_payout` pays a claim. UnderCover's rule is that a payout over $1,000 needs a person's approval, so the agent must never be able to make one. Before any rule exists, the gateway passes the call on and the app pays.

In [ ]:
await call_tool("approve_payout", claim_id="CLM-1002", amount=1150.00)

### The policy engine

AgentCore Policy checks every tool call at the gateway against rules written in Cedar, before the call reaches the app. A policy engine holds the rules. Its name takes underscores, since hyphens are not allowed.

In [ ]:
ENGINE_NAME = "undercover_policy_engine"


def find_policy_engines():
    pages = agentcore.get_paginator("list_policy_engines").paginate()
    return [e for page in pages for e in page["policyEngines"] if e["name"].startswith(PREFIX)]


existing = [e for e in find_policy_engines() if e["name"] == ENGINE_NAME]

if existing:
    engine_id = existing[0]["policyEngineId"]
else:
    engine_id = agentcore.create_policy_engine(
        name=ENGINE_NAME, description="Rules for the UnderCover gateway", tags=PROJECT_TAG
    )["policyEngineId"]

engine = wait_for(lambda: agentcore.get_policy_engine(policyEngineId=engine_id), f"Policy engine {engine_id}")
engine_arn = engine["policyEngineArn"]

The gateway's role needs permission to read the engine and to ask it for decisions.

In [ ]:
policy_engine_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "PolicyEngineConfiguration",
            "Effect": "Allow",
            "Action": "bedrock-agentcore:GetPolicyEngine",
            "Resource": engine_arn,
        },
        {
            "Sid": "PolicyEngineAuthorization",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:AuthorizeAction", "bedrock-agentcore:PartiallyAuthorizeActions"],
            "Resource": [engine_arn, gateway_arn],
        },
    ],
}

iam.put_role_policy(
    RoleName=ROLE_NAME, PolicyName="undercover-gateway-policy-engine", PolicyDocument=json.dumps(policy_engine_policy)
)
print("The role can use the policy engine")

### Attach it in `LOG_ONLY` mode

In `LOG_ONLY` mode the engine makes a decision on every call and records it, and the gateway lets the call through whatever the decision was. That is the safe way to try a rule on live traffic. AgentCore checks the role's new permission when the engine is attached, and the permission takes a few seconds to arrive, so the cell retries.

In [ ]:
def attach_policy_engine(mode):
    for attempt in range(20):
        try:
            agentcore.update_gateway(
                gatewayIdentifier=gateway_id,
                name=GATEWAY_NAME,
                description=gateway["description"],
                roleArn=role["Arn"],
                protocolType="MCP",
                authorizerType="AWS_IAM",
                exceptionLevel="DEBUG",
                policyEngineConfiguration={"arn": engine_arn, "mode": mode},
            )
            break
        except agentcore.exceptions.ValidationException as error:
            if "GetPolicyEngine" not in str(error):
                raise
            print("Waiting for the role's new permission to reach AgentCore")
            time.sleep(5)
    updated = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
    print(f"Policy engine mode: {updated['policyEngineConfiguration']['mode']}")


attach_policy_engine("LOG_ONLY")

### Two rules

With an engine attached, a tool call is refused unless a rule permits it. The first rule permits every tool of the `undercover-app` target to a caller who signed in with IAM. The second forbids a payout over $1,000, and a `forbid` wins over any `permit`.

`context.input` holds the tool's arguments. The amount is a JSON number, which Cedar reads as a decimal, so the comparison is `greaterThan` with `decimal(...)`. AgentCore checks each rule against the gateway's tools as it is created, so a misspelled tool name or argument fails here.

In [ ]:
policies = {
    "undercover_allow_app_tools": f"""permit(
  principal is AgentCore::IamEntity,
  action in AgentCore::Action::"{TARGET_NAME}",
  resource == AgentCore::Gateway::"{gateway_arn}"
);""",
    "undercover_payout_limit": f"""forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"{TARGET_NAME}___approve_payout",
  resource == AgentCore::Gateway::"{gateway_arn}"
)
when {{
  context.input.amount.greaterThan(decimal("1000.00"))
}};""",
}

print(policies["undercover_payout_limit"])

In [ ]:
def find_policies(engine_id):
    pages = agentcore.get_paginator("list_policies").paginate(policyEngineId=engine_id)
    return [p for page in pages for p in page["policies"]]


existing = {p["name"]: p for p in find_policies(engine_id)}

for name, statement in policies.items():
    if name in existing:
        policy_id = existing[name]["policyId"]
    else:
        policy_id = agentcore.create_policy(
            policyEngineId=engine_id, name=name, definition={"cedar": {"statement": statement}}
        )["policyId"]
    wait_for(lambda: agentcore.get_policy(policyEngineId=engine_id, policyId=policy_id), f"Policy {name}")

The engine is still in `LOG_ONLY` mode, so the payout over the limit goes through.

In [ ]:
await call_tool("approve_payout", claim_id="CLM-1002", amount=1150.00)

### Enforce

Now the same call is refused, and the message names the rule that refused it. A payout at the limit and a read still work.

In [ ]:
attach_policy_engine("ENFORCE")

await call_tool("approve_payout", claim_id="CLM-1002", amount=1150.00)
await call_tool("approve_payout", claim_id="CLM-1002", amount=1000.00)
await call_tool("get_claim", claim_id="CLM-1002")

The app logs a line for every payout it makes. The refused payout is missing from it, because the gateway stopped the call before it reached the app.

In [ ]:
time.sleep(5)
pages = logs.get_paginator("filter_log_events").paginate(
    logGroupName=f"/aws/lambda/{FUNCTION_NAME}", filterPattern="Paid", startTime=int((time.time() - 600) * 1000)
)

for page in pages:
    for event in page["events"]:
        print(time.strftime("%H:%M:%S", time.localtime(event["timestamp"] / 1000)), event["message"].strip())

The engine's decisions are counted in CloudWatch for each mode. The refusal made while in `LOG_ONLY` mode is there too, which is how a rule is checked before it is enforced. The numbers take a minute or two to arrive, so run the cell again if they are low.

In [ ]:
from datetime import datetime, timedelta, timezone

cloudwatch = session.client("cloudwatch")


def decisions(metric, mode):
    now = datetime.now(timezone.utc)
    stats = cloudwatch.get_metric_statistics(
        Namespace="AWS/Bedrock-AgentCore",
        MetricName=metric,
        Dimensions=[
            {"Name": "OperationName", "Value": "AuthorizeAction"},
            {"Name": "Mode", "Value": mode},
            {"Name": "PolicyEngine", "Value": engine_id},
        ],
        StartTime=now - timedelta(hours=3),
        EndTime=now,
        Period=3600,
        Statistics=["Sum"],
    )
    return int(sum(point["Sum"] for point in stats["Datapoints"]))


for mode in ("LOG_ONLY", "ENFORCE"):
    print(f"{mode:<10}allowed {decisions('AllowDecisions', mode)}, refused {decisions('DenyDecisions', mode)}")

## 8. What now exists in the account

Everything is found by name, then once more by the `project=undercover` tag across every service.

In [ ]:
def find_roles(prefix=PREFIX):
    pages = iam.get_paginator("list_roles").paginate()
    return [r["RoleName"] for page in pages for r in page["Roles"] if r["RoleName"].startswith(prefix)]


def find_secrets():
    pages = secrets_manager.get_paginator("list_secrets").paginate(IncludePlannedDeletion=True)
    return [s["Name"] for page in pages for s in page["SecretList"] if KEY_PROVIDER_NAME in s["Name"]]


def find_tagged():
    tag_filter = [{"Key": key, "Values": [value]} for key, value in PROJECT_TAG.items()]
    pages = tagging.get_paginator("get_resources").paginate(TagFilters=tag_filter)
    return [r["ResourceARN"] for page in pages for r in page["ResourceTagMappingList"]]


def show_resources():
    gateways = find_gateways(PREFIX)
    targets = [f"{g['name']}/{t['name']}" for g in gateways for t in find_targets(g["gatewayId"])]
    found = {
        "Gateways": [g["name"] for g in gateways],
        "Gateway targets": targets,
        "Policy engines": [e["name"] for e in find_policy_engines()],
        "API key providers": find_key_providers(),
        "Secrets": find_secrets(),
        "IAM roles": find_roles(),
        "Tagged project=undercover": find_tagged(),
    }
    for label, names in found.items():
        print(f"{label + ':':<28}{', '.join(names) or 'none'}")


show_resources()

## 9. Teardown (optional)

Run this when you are done. It deletes what this notebook made and leaves the Lambda app alone: the targets and then the gateway, the rules and then the policy engine, the credential provider with its secret, and the one role `undercover-gateway-role`.

First, what will be deleted.

In [ ]:
to_delete = {
    "Gateways": find_gateways(),
    "Policy engines": find_policy_engines(),
    "API key providers": [name for name in find_key_providers() if name == KEY_PROVIDER_NAME],
    "IAM roles": find_roles(ROLE_NAME),
}

for label, items in to_delete.items():
    names = [item["name"] if isinstance(item, dict) else item for item in items]
    print(f"{label + ':':<28}{', '.join(names) or 'none'}")

Then the delete. A gateway cannot be deleted while it has targets, and a policy engine cannot be deleted while it has rules, so those go first and the cell waits for them to clear.

In [ ]:
for gateway_summary in to_delete["Gateways"]:
    doomed_id = gateway_summary["gatewayId"]
    for target_summary in find_targets(doomed_id):
        agentcore.delete_gateway_target(gatewayIdentifier=doomed_id, targetId=target_summary["targetId"])
        print(f"Deleted target {target_summary['name']}")
    while find_targets(doomed_id):
        time.sleep(3)
    agentcore.delete_gateway(gatewayIdentifier=doomed_id)
    print(f"Deleted gateway {gateway_summary['name']}")

for engine_summary in to_delete["Policy engines"]:
    doomed_id = engine_summary["policyEngineId"]
    for policy_summary in find_policies(doomed_id):
        agentcore.delete_policy(policyEngineId=doomed_id, policyId=policy_summary["policyId"])
        print(f"Deleted policy {policy_summary['name']}")
    while find_policies(doomed_id):
        time.sleep(3)
    agentcore.delete_policy_engine(policyEngineId=doomed_id)
    print(f"Deleted policy engine {engine_summary['name']}")

for name in to_delete["API key providers"]:
    agentcore.delete_api_key_credential_provider(name=name)
    print(f"Deleted credential provider {name}")

for name in to_delete["IAM roles"]:
    for policy_name in iam.list_role_policies(RoleName=name)["PolicyNames"]:
        iam.delete_role_policy(RoleName=name, PolicyName=policy_name)
    iam.delete_role(RoleName=name)
    print(f"Deleted role {name}")

The same searches again. AgentCore removes the secret a few seconds after its credential provider, so the cell waits for that. The Lambda app and its role should be all that is left.

In [ ]:
deadline = time.time() + 60
while (find_gateways() or find_policy_engines() or find_secrets() or any(":gateway/" in arn for arn in find_tagged())) and time.time() < deadline:
    time.sleep(3)

show_resources()